# 1. Data, labels, and tokens
The archived UNSW file contains ordered payload bytes and an attack-category `label`. We map benign/normal to 0 and all attack categories to 1. Labels are never included in tokens.

In [ ]:
from pathlib import Path
from benignids.config import load_config
from benignids.data import load_payload_dataset, make_binary_target
from benignids.tokenization import TokenizerConfig, TrafficTokenizer
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
config = load_config(PROJECT_ROOT / 'configs/default.yaml')
Xy = load_payload_dataset(config['data']['path'], payload_prefix_bytes=256, sample_rows=5000)
y = make_binary_target(Xy['label'], config['data']['benign_labels'])
X = Xy.drop(columns=['label'])
print(X.shape, y.value_counts(normalize=True).sort_index())

In [ ]:
tokenizer = TrafficTokenizer(TokenizerConfig(max_length=256, payload_prefix_bytes=256))
input_ids, attention_mask = tokenizer.encode_frame(X.head(8))
print(input_ids.shape, attention_mask.sum(axis=1))

## Leakage check
The deterministic tokenizer has no `fit(X_test)` step. `label` and `attack_cat` are excluded, and ordered `payload_byte_N` fields become dedicated byte tokens.